# Agentic RAG Patterns: Corrective RAG (CRAG) & Self-RAG

**References:**
- Slide 10 (*Corrective RAG (CRAG)*) - Building Agentic Workflows
- Slide 11 (*Self-RAG Reflection Tokens*) - Building Agentic Workflows
- Slide 35 (*Agentic Loops & Query Rewriting*)

### Overview
Static RAG blindly augments LLMs with whatever chunks vector search returned, frequently passing irrelevant distractors into the context.

In this notebook, we implement two state-of-the-art agentic correction patterns:
1. **Corrective RAG (CRAG)**: Evaluates chunks (`CORRECT`, `INCORRECT`, `AMBIGUOUS`). If confidence drops below threshold or chunks are irrelevant, triggers an automated **Query Rewriting Loop** before final synthesis.
2. **Self-RAG Reflection Tokens**: Uses explicit self-reflection tokens to evaluate:
   - `[RETRIEVE]`: Was retrieval necessary?
   - `[IS-REL]`: Is the retrieved context relevant?
   - `[IS-SUP]`: Is the generated response factually supported / grounded?
   - `[IS-USE]`: Is the answer useful to the user?

In [ ]:
from crag_and_self_rag import CorrectiveRAGPipeline, SelfRAGEvaluator

pipeline = CorrectiveRAGPipeline(confidence_threshold=0.70, mock_mode=True)
self_rag = SelfRAGEvaluator(mock_mode=True)
print("Initialized CRAG Pipeline and Self-RAG Evaluator.")

## 1. High Relevance Scenario: Direct Synthesis
When retrieved chunks are relevant, CRAG proceeds directly without query rewriting.

In [ ]:
query = "What is the context window for Claude 3.5 Sonnet in Bedrock?"
chunks = [
    "Claude 3.5 Sonnet supports a 200,000 token context window on Amazon Bedrock.",
    "Bedrock models including Sonnet 3.5 handle up to 200k tokens per request."
]

result = pipeline.run(query, chunks)
print(f"Initial Confidence Score: {result['initial_confidence']}")
print(f"Was Query Rewritten: {result['query_rewritten']}")

# Verify with Self-RAG
verification = self_rag.verify_response(query, chunks[0], result["answer"])
print("\nSelf-RAG Tokens:", verification["tokens"])
print("Grounding Status:", verification["grounding_status"])

## 2. Low Relevance / Distractor Scenario: Automated Query Rewriting Loop
When retrieved chunks fail relevance evaluation, the CRAG pipeline automatically triggers the Query Rewriter.

In [ ]:
query_poor = "Configure cross region inference profiles in Bedrock"
distractor_chunks = [
    "Amazon S3 Glacier Flexible Retrieval offers low-cost cold storage.",
    "VPC peering connections allow private CIDR block routing."
]

result_poor = pipeline.run(query_poor, distractor_chunks)
print(f"Initial Confidence: {result_poor['initial_confidence']}")
print(f"Query Rewritten: {result_poor['query_rewritten']}")
print(f"Reformulated Query: '{result_poor['rewritten_query']}'")
print(f"Active Chunks After CRAG Loop: {result_poor['active_chunks_count']}")

## 3. Summary of Agentic Self-Correction Benefits
- Prevents model hallucination from distracting context chunks.
- Dynamically recovers when user queries are underspecified or terminology diverges from indexed vector vocabulary.
- Provides clear audit trails via Self-RAG reflection tokens.